In [1]:
import os
import geopandas as gpd
import pandas as pd
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from rasterio.plot import show
from datetime import datetime, timedelta
import ipywidgets as widgets
from IPython.display import display

# Set the working directory
os.chdir("/home/jovyan/work/Typhoon_IBF_Rice_Damage_Model/")
cdir = os.getcwd()

# Load the typhoon tracks shapefile
tracks_fp = 'IBF_typhoon_model/data/gis_data/typhoon_tracks/tracks_filtered.shx'
typhoon_tracks = gpd.read_file(tracks_fp)

# Set the CRS for the tracks shapefile to match the TIFF files (EPSG:4326)
typhoon_tracks = typhoon_tracks.set_crs(epsg=4326, allow_override=True)

# Create the `name_year` column by combining `NAME` and `year`
typhoon_tracks['name_year'] = typhoon_tracks['NAME'] + typhoon_tracks['year'].astype(str)

# Convert `ISO_TIME` to datetime
typhoon_tracks['ISO_TIME'] = pd.to_datetime(typhoon_tracks['ISO_TIME'], errors='coerce')

# Print all unique `name_year` combinations
unique_name_years = typhoon_tracks['name_year'].unique()
print("Available storms (name_year combinations):")
for name_year in unique_name_years:
    print(name_year)

# Load the administrative boundaries shapefile
admin_boundaries_fp = 'IBF_typhoon_model/data/phl_administrative_boundaries/country_boundary/PHL_adm0.shp'
admin_boundaries = gpd.read_file(admin_boundaries_fp)
admin_boundaries = admin_boundaries.set_crs(epsg=4326, allow_override=True)

# Verify admin boundaries CRS
print("Admin boundaries CRS:", admin_boundaries.crs)

# Get available storms by listing folders in the GPM directory
gpm_base_folder = 'IBF_typhoon_model/data/rainfall_data/output_hhr/gpm/'
available_storms = [storm for storm in os.listdir(gpm_base_folder) if os.path.isdir(os.path.join(gpm_base_folder, storm))]
print("Available storms:", available_storms)

# Specify the typhoon name and year to visualize
typhoon_name_year = "NOCK-TEN2016"  # Replace this with the desired typhoon name and year

# Function to update visualization based on specified storm
def update_visualization(storm):
    print(f"Selected storm: {storm}")
    
    # Filter tracks based on specified storm
    filtered_tracks = typhoon_tracks[typhoon_tracks['name_year'] == storm]
    
    if filtered_tracks.empty:
        print(f"No tracks found for storm: {storm}")
        return
    
    # Check for NaT values in ISO_TIME
    if filtered_tracks['ISO_TIME'].isna().all():
        print(f"All ISO_TIME values are NaT for storm: {storm}")
        return
    
    # Determine the start and end times from the tracks
    start_time = filtered_tracks['ISO_TIME'].min()
    end_time = filtered_tracks['ISO_TIME'].max()
    
    if pd.isna(start_time) or pd.isna(end_time):
        print(f"Invalid start or end time for storm: {storm}")
        return
    
    # Generate 3-hour timesteps between start_time and end_time
    timestep = timedelta(hours=3)
    time_range = pd.date_range(start=start_time, end=end_time, freq=timestep)
    
    print(f"Available visualization period: {start_time} to {end_time}")
    print(f"3-hour timesteps available for visualization:")
    for time in time_range:
        print(time)
    
    # Use the provided bounding box values
    bbox = [80.150, -0.568, 180, 54.532]
    print("Bounding box:", bbox)
    
    # Convert the storm name to lowercase for the GPM folder path
    storm_lowercase = storm.lower()
    gpm_folder = os.path.join(gpm_base_folder, storm_lowercase, 'GPM')
    tif_files = []

    try:
        for date_folder in os.listdir(gpm_folder):
            date_folder_path = os.path.join(gpm_folder, date_folder)
            if os.path.isdir(date_folder_path):
                for tif_file in os.listdir(date_folder_path):
                    if tif_file.endswith('.tif'):
                        tif_fp = os.path.join(date_folder_path, tif_file)
                        base_name = os.path.basename(tif_fp)
                        
                        # Extract date and time from GPM filename, remove 'S' character
                        date_str = base_name.split('.')[4][:8]
                        time_str = base_name.split('.')[4][9:15].replace('S', '')  # Remove 'S'
                        datetime_str = datetime.strptime(f"{date_str} {time_str}", '%Y%m%d %H%M%S')
                        tif_files.append((datetime_str, tif_fp, date_str, time_str))
    except FileNotFoundError:
        print(f"GPM data folder not found for storm: {storm}")
        return
    
    # Sort the TIFF files by datetime
    tif_files.sort()

    # Generate a list of datetime strings for the slider
    datetime_strings = [dt.strftime('%Y-%m-%d %H:%M:%S') for dt, _, _, _ in tif_files]

    # Check the datetime strings
    print("Datetime strings:", datetime_strings[:10])

    # Create interactive slider for visualization
    slider = widgets.SelectionSlider(
        options=datetime_strings,
        description='DateTime:',
        disabled=False,
        continuous_update=False,
        orientation='horizontal',
        readout=True
    )

    # Function to visualize a timestep
    def visualize_timestep(date_time_str):
        datetime_obj = datetime.strptime(date_time_str, '%Y-%m-%d %H:%M:%S')
        print(f"Visualizing for {datetime_obj}")

        for dt, tif_fp, _, _ in tif_files:
            if dt == datetime_obj:
                # Open the tif file
                print(f"Opening TIFF file: {tif_fp}")
                with rasterio.open(tif_fp) as src:
                    fig, ax = plt.subplots(figsize=(40, 40))
                    
                    # Read the raster data
                    data = src.read(1)
                    print(f"Raster data shape: {data.shape}")
                    
                    # Apply cumulative count cut to 97%
                    valid_data = data[data > 0]
                    cut_value = np.percentile(valid_data, 95)
                    data[data > cut_value] = cut_value
                    print(f"Cut value for 95th percentile: {cut_value}")
                    
                    show(data, transform=src.transform, ax=ax, cmap='viridis')
                    
                    # Plot the track for the specific time
                    track_for_time = filtered_tracks[filtered_tracks['ISO_TIME'] == datetime_obj]
                    print(f"Track for time shape: {track_for_time.shape}")
                    if not track_for_time.empty:
                        track_for_time.plot(ax=ax, color='red', linewidth=1)
                        for _, row in track_for_time.iterrows():
                            if row['USA_ROCI'] > 0:
                                centroid = row.geometry.centroid
                                roci_km = row['USA_ROCI'] * 1.852  # Convert nautical miles to kilometers
                                create_circle_km(ax, centroid, roci_km, color='blue', alpha=0.3)
                    else:
                        print(f"No tracks found for {datetime_obj}")

                    # Plot administrative boundaries
                    admin_boundaries.plot(ax=ax, facecolor='none', edgecolor='blue')

                    # Set plot bounds
                    ax.set_xlim(bbox[0], bbox[2])
                    ax.set_ylim(bbox[1], bbox[3])
                    
                    plt.title(f"Precipitation and Track for {datetime_obj}")
                    plt.show()
                break

    # Button click functions
    def on_next_button_clicked(b):
        current_index = slider.options.index(slider.value)
        if current_index < len(slider.options) - 1:
            slider.value = slider.options[current_index + 1]

    def on_previous_button_clicked(b):
        current_index = slider.options.index(slider.value)
        if current_index > 0:
            slider.value = slider.options[current_index - 1]

    # Create buttons for moving the slider
    next_button = widgets.Button(description="Next")
    previous_button = widgets.Button(description="Previous")

    next_button.on_click(on_next_button_clicked)
    previous_button.on_click(on_previous_button_clicked)

    # Display the slider and buttons
    display(slider)
    display(previous_button, next_button)

    # Link the slider to the visualization function
    interactive_plot = widgets.interactive(visualize_timestep, date_time_str=slider)
    display(interactive_plot)

# Function to create a circle in kilometers around a point
def create_circle_km(ax, center_point, radius_km, **kwargs):
    lat_radius = radius_km / 111  # Convert km to degrees of latitude
    circle = plt.Circle((center_point.x, center_point.y), lat_radius, **kwargs)
    ax.add_patch(circle)

# Update visualization based on specified storm
update_visualization(typhoon_name_year)


ModuleNotFoundError: No module named 'rasterio'

In [7]:
import geopandas as gpd


# Load the typhoon tracks shapefile
tracks_fp = 'IBF_typhoon_model/data/gis_data/typhoon_tracks/tracks_filtered.shx'
typhoon_tracks = gpd.read_file(tracks_fp)

# Print the column names
print("Column names in the typhoon_tracks GeoDataFrame:")
print(typhoon_tracks.columns)

# Display 5 random entries
print("\nSample entries from typhoon_tracks GeoDataFrame:")
print(typhoon_tracks.sample(5))  


Column names in the typhoon_tracks GeoDataFrame:
Index(['SID', 'SEASON', 'NUMBER', 'BASIN', 'SUBBASIN', 'NAME', 'ISO_TIME',
       'NATURE', 'LAT', 'LON',
       ...
       'USA_SEA_SW', 'USA_SEA_NW', 'STORM_SPD', 'STORM_DR', 'year', 'month',
       'day', 'hour', 'min', 'geometry'],
      dtype='object', length=169)

Sample entries from typhoon_tracks GeoDataFrame:
                SID  SEASON  NUMBER BASIN SUBBASIN       NAME  \
407   2010240N15142    2010      49    WP       MM    KOMPASU   
2286  2015285N14151    2015      92    WP       MM      KOPPU   
1769  2014260N13135    2014      69    WP       MM  FUNG-WONG   
1830  2014334N02156    2014      88    WP       MM    HAGUPIT   
280   2009270N10148    2009      74    WP       MM      PARMA   

                 ISO_TIME NATURE    LAT     LON  ...  USA_SEA_SW  USA_SEA_NW  \
407   2010-08-31 18:00:00     TS  28.68  126.20  ...         NaN         NaN   
2286  2015-10-13 18:00:00     TS  15.82  137.57  ...         NaN         NaN   
